# Capstone 4: Mental Health Concern Classifier

This notebook implements the required four-model comparison. It uses one stratified 80/20 split (seed 42), selects by validation macro-F1 with accuracy as a tie-breaker, and creates `submission.csv` only after model selection. Predictions are research/triage signals, not clinical diagnoses.

**Runtime:** Run in the course Databricks Serverless GPU environment. The local capstone folder does not include its pretrained model assets or course-installed ML libraries.


## 1. Setup and reproducibility

The data loader accepts the course filenames (`.csv`) and the local copies (`.csv.txt`). Model assets are loaded only from the documented local workspace paths; no internet download is attempted. Credentials are read from Databricks secrets, never displayed.


In [ ]:
# If required by the course workspace, run this in a separate Databricks cell:
# %run ./.setup/learner_setup

import os, re, sys, json, time, random, inspect, warnings
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# Optional framework seeds (available in the course runtime).
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
except ImportError:
    torch = None

print("Python:", sys.version.split()[0], "| NumPy:", np.__version__, "| pandas:", pd.__version__)


## 2. Load data and create the shared split

Assertions catch schema changes early. The test rows remain in their original order and are not used to fit any vocabulary, embedding, or model.


In [ ]:
def resolve_file(stem):
    candidates = [Path(stem + ".csv"), Path(stem + ".csv.txt")]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {candidates[0]} or {candidates[1]} in {Path.cwd()}")

train_path = resolve_file("cap4_train_val")
test_path = resolve_file("cap4_test")
train_val_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
assert list(train_val_df.columns) == ["text", "label"]
assert list(test_df.columns) == ["text"]
assert train_val_df["text"].notna().all() and test_df["text"].notna().all()
assert train_val_df["text"].astype(str).str.strip().ne("").all()
assert test_df["text"].astype(str).str.strip().ne("").all()
assert set(train_val_df["label"].unique()) == {0, 1}

print("Train/validation source:", train_path, train_val_df.shape)
print("Test source:", test_path, test_df.shape)
print("Label counts:\n", train_val_df.label.value_counts().sort_index())
print("Word-count summary:\n", train_val_df.text.str.split().str.len().describe(percentiles=[.5, .9, .95, .99]))
print("Exact duplicate texts in labeled data:", int(train_val_df.text.duplicated().sum()))
print("Exact duplicate texts in test data:", int(test_df.text.duplicated().sum()))

from sklearn.model_selection import train_test_split
all_positions = np.arange(len(train_val_df))
train_idx, val_idx = train_test_split(
    all_positions, test_size=0.20, random_state=SEED,
    stratify=train_val_df.label.to_numpy()
)
train_text = train_val_df.iloc[train_idx].text.astype(str).tolist()
val_text = train_val_df.iloc[val_idx].text.astype(str).tolist()
y_train = train_val_df.iloc[train_idx].label.to_numpy(dtype=int)
y_val = train_val_df.iloc[val_idx].label.to_numpy(dtype=int)
test_text = test_df.text.astype(str).tolist()
print(f"Split: {len(train_idx)} train / {len(val_idx)} validation; seed={SEED}")
print("Train labels:", dict(zip(*np.unique(y_train, return_counts=True))))
print("Validation labels:", dict(zip(*np.unique(y_val, return_counts=True))))
train_text_set = set(train_text)
val_text_set = set(val_text)
print("Exact text overlap across train/validation (report only):", len(train_text_set & val_text_set))


## 3. Model-specific preprocessing

Classical models use light normalization and URL/email/handle masking. The LLM receives readable natural language with whitespace normalization only. DistilBERT receives the original text with minimal cleanup. Stop-word and punctuation deletion are avoided because these may carry meaning in social posts.


In [ ]:
def clean_classical(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " URLTOKEN ", text)
    text = re.sub(r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b", " EMAILTOKEN ", text)
    text = re.sub(r"(?<!\w)@[A-Za-z0-9_]+", " HANDLETOKEN ", text)
    return re.sub(r"\s+", " ", text).strip()

def clean_prompt(text):
    return re.sub(r"\s+", " ", str(text)).strip()

train_classical = [clean_classical(x) for x in train_text]
val_classical = [clean_classical(x) for x in val_text]
test_classical = [clean_classical(x) for x in test_text]
train_prompt = [clean_prompt(x) for x in train_text]
val_prompt = [clean_prompt(x) for x in val_text]
test_prompt = [clean_prompt(x) for x in test_text]


## 4. Shared evaluation helpers

All validation comparisons use macro-F1 as the primary metric and accuracy as secondary. Confusion matrices use label order `[0, 1]`.


In [ ]:
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report
from IPython.display import display

results = {}
def evaluate(name, y_true, y_pred, settings=None):
    y_pred = np.asarray(y_pred, dtype=int)
    assert len(y_pred) == len(y_true) and set(np.unique(y_pred)).issubset({0, 1})
    metrics = {
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist(),
        "classification_report": classification_report(y_true, y_pred, labels=[0, 1], target_names=["0 no concern", "1 potential concern"], zero_division=0, output_dict=True),
        "settings": settings or {},
        "predictions": y_pred,
    }
    results[name] = metrics
    print(f"{name}: macro-F1={metrics['macro_f1']:.4f} | accuracy={metrics['accuracy']:.4f}")
    print("Confusion matrix [true rows, predicted columns; labels 0,1]:\n", np.array(metrics["confusion_matrix"]))
    display(pd.DataFrame(metrics["classification_report"]).T.round(3))
    return metrics


## 5. Classical NLP - TF-IDF + Logistic Regression

A small validation-only sweep covers unigrams/bigrams, document-frequency thresholds, and regularization. Each pipeline fits its vectorizer only on the training split.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf_candidates = []
for ngram_range, min_df, max_df, C in [
    ((1, 1), 2, 0.98, 1.0),
    ((1, 2), 2, 0.98, 1.0),
    ((1, 2), 3, 0.95, 3.0),
    ((1, 3), 2, 0.98, 1.0),
]:
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=ngram_range, min_df=min_df, max_df=max_df, sublinear_tf=True)),
        ("classifier", LogisticRegression(C=C, max_iter=1500, random_state=SEED, class_weight=None)),
    ])
    pipe.fit(train_classical, y_train)
    pred = pipe.predict(val_classical)
    name = f"TF-IDF ngram={ngram_range}, min_df={min_df}, max_df={max_df}, C={C}"
    m = evaluate(name, y_val, pred, {"ngram_range": ngram_range, "min_df": min_df, "max_df": max_df, "C": C})
    tfidf_candidates.append((m["macro_f1"], m["accuracy"], pipe, name))

best_tfidf = max(tfidf_candidates, key=lambda x: (x[0], x[1]))
tfidf_model, tfidf_name = best_tfidf[2], best_tfidf[3]
print("Selected TF-IDF variant by validation macro-F1, then accuracy:", tfidf_name)


## 6. Classical NLP - Word2Vec average embeddings + Logistic Regression

Word2Vec is trained on the training documents only. Document vectors are arithmetic means of in-vocabulary token vectors; empty/OOV-only documents receive zero vectors and are counted.


In [ ]:
from gensim.models import Word2Vec
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

def tokenize_classical(text):
    return re.findall(r"[a-z0-9]+|(?:URLTOKEN|EMAILTOKEN|HANDLETOKEN)", text)

def average_vectors(model, texts):
    vectors, empty = [], 0
    for text in texts:
        tokens = [tok for tok in tokenize_classical(text) if tok in model.wv]
        if tokens:
            vectors.append(np.mean(model.wv[tokens], axis=0))
        else:
            vectors.append(np.zeros(model.vector_size, dtype=np.float32))
            empty += 1
    return np.asarray(vectors, dtype=np.float32), empty

w2v_candidates = []
train_tokens = [tokenize_classical(x) for x in train_classical]
for vector_size, window, epochs in [(100, 5, 15), (200, 10, 20)]:
    w2v = Word2Vec(
        sentences=train_tokens, vector_size=vector_size, window=window, min_count=1,
        workers=1, sg=1, epochs=epochs, seed=SEED,
    )
    X_train, empty_train = average_vectors(w2v, train_classical)
    X_val, empty_val = average_vectors(w2v, val_classical)
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1500, random_state=SEED))
    clf.fit(X_train, y_train)
    pred = clf.predict(X_val)
    name = f"Word2Vec+LogReg dim={vector_size}, window={window}, epochs={epochs}"
    settings = {"vector_size": vector_size, "window": window, "min_count": 1, "epochs": epochs, "sg": 1, "zero_vectors_train": empty_train, "zero_vectors_validation": empty_val}
    m = evaluate(name, y_val, pred, settings)
    w2v_candidates.append((m["macro_f1"], m["accuracy"], w2v, clf, name, settings))

best_w2v = max(w2v_candidates, key=lambda x: (x[0], x[1]))
w2v_model, w2v_classifier, w2v_name, w2v_settings = best_w2v[2], best_w2v[3], best_w2v[4], best_w2v[5]
print("Selected Word2Vec variant:", w2v_name)


## 7. Advanced NLP - prompted Azure LLM classifier

Uses the configured Azure `gpt-4.1-mini` deployment, temperature 0, and up to three balanced examples per class sampled from the training split only. The validation pass is intentionally explicit because it may require 300 API calls. Responses are accepted only when they contain a single `0` or `1`; malformed responses get one constrained retry and are then recorded as errors.

Credentials must be available in Databricks secrets (`AIML_Training/client_id` and `AIML_Training/client_sec`). Never print credentials or access tokens.


In [ ]:
import httpx
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI

# Resolve either the course filename or the local copy. In Databricks, place the file in the working directory.
for env_path in [Path("UAIS_vars.env"), Path("UAIS_vars.env.txt")]:
    if env_path.exists():
        load_dotenv(env_path, override=False)
        break
required_env = ["AZURE_OPENAI_ENDPOINT", "MODEL_DEPLOYMENT_NAME", "OPENAI_API_VERSION", "PROJECT_ID"]
missing_env = [key for key in required_env if not os.environ.get(key)]
if missing_env:
    raise EnvironmentError(f"Missing Azure configuration keys: {missing_env}")

def get_access_token():
    if "dbutils" not in globals():
        raise RuntimeError("Azure LLM calls require the course Databricks dbutils secrets.")
    response = httpx.post(
        "https://api.uhg.com/oauth2/token",
        data={
            "grant_type": "client_credentials",
            "scope": "https://api.uhg.com/.default",
            "client_id": dbutils.secrets.get(scope="AIML_Training", key="client_id"),
            "client_secret": dbutils.secrets.get(scope="AIML_Training", key="client_sec"),
        },
        headers={"Content-Type": "application/x-www-form-urlencoded"}, timeout=60
    )
    response.raise_for_status()
    return response.json()["access_token"]

llm_client = AzureChatOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_version=os.environ["OPENAI_API_VERSION"],
    azure_deployment=os.environ["MODEL_DEPLOYMENT_NAME"],
    temperature=0, azure_ad_token=get_access_token(),
    default_headers={"projectId": os.environ["PROJECT_ID"], "model-usage-type": "prod"},
)

# Sample examples deterministically from training only; never from validation or test.
train_examples = train_val_df.iloc[train_idx][["text", "label"]]
shots = []
for label in [0, 1]:
    pool = train_examples[train_examples.label == label]
    shots.extend(pool.sample(n=min(3, len(pool)), random_state=SEED + label).to_dict("records"))

def make_prompt(text, retry=False):
    examples = "\n".join(f"Text: {clean_prompt(row['text'])}\nLabel: {int(row['label'])}" for row in shots)
    extra = " Return exactly one character: 0 or 1. No explanation." if retry else ""
    return (
        "You classify short social-media posts for possible mental-health concern triage. This is not a diagnosis.\n"
        "Label 1 means the post indicates a potential anxiety or stress concern. Label 0 means no such concern is detected.\n"
        "Judge the text itself; do not infer a diagnosis from identity or unrelated details. If evidence is absent, choose 0.\n"
        f"Examples from labeled training data only:\n{examples}\n"
        f"Classify this post. Output only the single digit 0 or 1.{extra}\nText: {clean_prompt(text)}\nLabel:"
    )

def parse_label(content):
    match = re.fullmatch(r"\s*([01])\s*", str(content))
    return int(match.group(1)) if match else None

def llm_predict(texts):
    predictions, malformed = [], []
    for i, text in enumerate(texts):
        value = None
        for attempt in range(2):
            try:
                response = llm_client.invoke(make_prompt(text, retry=(attempt == 1)))
                value = parse_label(response.content)
                if value is not None:
                    break
            except Exception as exc:
                if attempt == 1:
                    malformed.append((i, type(exc).__name__))
            time.sleep(min(1.0 * (attempt + 1), 2.0))
        if value is None:
            malformed.append((i, "unparseable after retry"))
            raise ValueError(f"LLM failed strict parsing at row {i}; inspect and rerun this validation pass.")
        predictions.append(value)
        if (i + 1) % 50 == 0:
            print(f"LLM progress: {i+1}/{len(texts)}")
    return np.asarray(predictions, dtype=int), malformed

llm_val_pred, llm_errors = llm_predict(val_prompt)
llm_metrics = evaluate("Azure gpt-4.1-mini prompt classifier", y_val, llm_val_pred, {"few_shot_per_class": 3, "temperature": 0, "deployment": os.environ["MODEL_DEPLOYMENT_NAME"]})
print("Malformed/retried outputs:", len(llm_errors))


## 8. Advanced NLP - fine-tune local DistilBERT

The pretrained tokenizer and checkpoint are loaded from the course's offline local path. This cell uses truncation, a 256-token starting limit (covering the great majority of observed posts), dynamic padding, and a best-validation-macro-F1 checkpoint. The length can be adjusted only using training/validation evidence and must remain <= 512.


In [ ]:
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments, set_seed
)
from sklearn.metrics import f1_score, accuracy_score

MODEL_DIR = "/Workspace/advanced_data/distilbert-model"
assert Path(MODEL_DIR).exists(), f"DistilBERT assets not found at {MODEL_DIR}; run in course Databricks environment."
MAX_LENGTH = 256
EPOCHS = 3
BATCH_SIZE = 16
LEARNING_RATE = 3e-5
set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)

def make_hf_dataset(texts, labels=None):
    data = {"text": texts}
    if labels is not None:
        data["labels"] = [int(x) for x in labels]
    dataset = Dataset.from_dict(data)
    def tokenize_batch(batch):
        return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)
    return dataset.map(tokenize_batch, batched=True, remove_columns=["text"])

train_ds = make_hf_dataset(train_text, y_train)
val_ds = make_hf_dataset(val_text, y_val)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR, num_labels=2, local_files_only=True
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"f1": f1_score(labels, preds, average="macro"), "accuracy": accuracy_score(labels, preds)}

training_kwargs = dict(
    output_dir="/tmp/capstone4-distilbert-checkpoints",
    learning_rate=LEARNING_RATE, per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=32, num_train_epochs=EPOCHS, weight_decay=0.01,
    logging_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="f1", greater_is_better=True,
    save_total_limit=1, report_to="none", seed=SEED, data_seed=SEED,
    fp16=bool(torch and torch.cuda.is_available()),
)
arg_params = inspect.signature(TrainingArguments.__init__).parameters
training_kwargs["eval_strategy" if "eval_strategy" in arg_params else "evaluation_strategy"] = "epoch"
training_args = TrainingArguments(**training_kwargs)
trainer = Trainer(
    model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds,
    tokenizer=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics,
)
trainer.train()
dbert_output = trainer.predict(val_ds)
dbert_val_pred = np.argmax(dbert_output.predictions, axis=-1).astype(int)
dbert_metrics = evaluate(
    "Fine-tuned DistilBERT", y_val, dbert_val_pred,
    {"max_length": MAX_LENGTH, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE}
)


## 9. Compare results and select the final approach

Model selection uses validation macro-F1 first and accuracy second. The test set is still untouched at this point. Review per-class recall and false-positive/false-negative counts before freezing the selection.


In [ ]:
summary_rows = []
for name, metrics in results.items():
    summary_rows.append({"model": name, "validation_macro_f1": metrics["macro_f1"], "validation_accuracy": metrics["accuracy"], "settings": json.dumps(metrics["settings"], default=str)})
comparison = pd.DataFrame(summary_rows).sort_values(["validation_macro_f1", "validation_accuracy"], ascending=False).reset_index(drop=True)
display(comparison)
best_name = comparison.iloc[0].model
print("Selected by validation macro-F1, then accuracy:", best_name)
print("Interpretation prompt: discuss the error balance and tradeoffs shown in the table and reports. Do not claim diagnosis or hidden-test performance.")


## 10. Refit selected model on all labeled data and predict test

After selection is frozen, the selected classical or DistilBERT configuration is refit using all 1,500 labeled rows. If the LLM wins, use the frozen prompt and examples from the original training split. Test text is used only for final prediction.


In [ ]:
all_text_raw = train_val_df.text.astype(str).tolist()
all_y = train_val_df.label.to_numpy(dtype=int)
all_text_classical = [clean_classical(x) for x in all_text_raw]
all_text_prompt = [clean_prompt(x) for x in all_text_raw]

if best_name.startswith("TF-IDF"):
    selected_model = next(item[2] for item in tfidf_candidates if item[3] == best_name)
    selected_model.fit(all_text_classical, all_y)
    test_predictions = selected_model.predict(test_classical).astype(int)
elif best_name.startswith("Word2Vec"):
    selected = next(item for item in w2v_candidates if item[4] == best_name)
    cfg = selected[5]
    final_w2v = Word2Vec(
        sentences=[tokenize_classical(x) for x in all_text_classical],
        vector_size=cfg["vector_size"], window=cfg["window"], min_count=cfg["min_count"],
        workers=1, sg=cfg["sg"], epochs=cfg["epochs"], seed=SEED,
    )
    X_all, _ = average_vectors(final_w2v, all_text_classical)
    X_test, final_oov = average_vectors(final_w2v, test_classical)
    final_clf = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1500, random_state=SEED))
    final_clf.fit(X_all, all_y)
    test_predictions = final_clf.predict(X_test).astype(int)
    print("All-OOV/empty test documents:", final_oov)
elif best_name == "Azure gpt-4.1-mini prompt classifier":
    test_predictions, test_llm_errors = llm_predict(test_prompt)
    print("Malformed/retried test outputs:", len(test_llm_errors))
elif best_name == "Fine-tuned DistilBERT":
    # Reinitialize and fit with the selected validation configuration on all labels.
    final_train_ds = make_hf_dataset(all_text_raw, all_y)
    final_test_ds = make_hf_dataset(test_text)
    final_model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, num_labels=2, local_files_only=True)
    final_args_kwargs = dict(training_kwargs)
    final_args_kwargs["output_dir"] = "/tmp/capstone4-distilbert-final"
    final_args_kwargs["num_train_epochs"] = EPOCHS
    final_args_kwargs["load_best_model_at_end"] = False
    final_args_kwargs["save_strategy"] = "no"
    final_args_kwargs.pop("eval_strategy", None)
    final_args_kwargs.pop("evaluation_strategy", None)
    final_args = TrainingArguments(**final_args_kwargs)
    final_trainer = Trainer(model=final_model, args=final_args, train_dataset=final_train_ds, tokenizer=tokenizer, data_collator=data_collator)
    final_trainer.train()
    final_output = final_trainer.predict(final_test_ds)
    test_predictions = np.argmax(final_output.predictions, axis=-1).astype(int)
else:
    raise RuntimeError(f"Unrecognized selected model: {best_name}")

assert len(test_predictions) == len(test_df)
assert set(np.unique(test_predictions)).issubset({0, 1})


## 11. Validate and save `submission.csv`

The deliverable must contain exactly one integer column named `y_pred`, no index, and one prediction per test row in original order. Test labels are unavailable, so no test score can be calculated here.


In [ ]:
submission = pd.DataFrame({"y_pred": np.asarray(test_predictions, dtype=int)})
assert list(submission.columns) == ["y_pred"]
assert len(submission) == len(test_df)
assert submission.y_pred.notna().all() and submission.y_pred.isin([0, 1]).all()
submission_path = Path("submission.csv")
submission.to_csv(submission_path, index=False)
check = pd.read_csv(submission_path)
assert list(check.columns) == ["y_pred"] and len(check) == len(test_df)
assert check.y_pred.isin([0, 1]).all()
print(f"Saved {submission_path.resolve()} with {len(check)} predictions.")
print("Prediction counts:", check.y_pred.value_counts().sort_index().to_dict())


## 12. Final checklist

- [ ] All four approaches ran and have validation macro-F1, accuracy, confusion matrix, and classification report.
- [ ] One shared stratified 80/20 split with seed 42 was reused.
- [ ] Vectorizer and Word2Vec were fitted only on the training split before model selection.
- [ ] LLM few-shot examples came only from training, with no more than five per class.
- [ ] DistilBERT loaded locally, used truncation and dynamic padding, and trained only on training data for validation.
- [ ] Best approach was selected by validation macro-F1, accuracy secondary.
- [ ] Test predictions were generated only after selection and were saved as one-column `submission.csv`.
- [ ] All notebook cells have been executed successfully and outputs are retained.
- [ ] Databricks compute was terminated after submission.
